# Table 4: Proportion of NXDOMAIN responses
Reproduce all four Table 4 columns from saved response-status summaries in `script/res/`, without rerunning measurement scripts.

`NXDOMAIN percentage = 100 × NXDOMAIN count / total queries`

Inputs:
- Source: `20260406_nxdomain_ratio_stats.csv`
- WWW: `20260406_www_nxdomain_ratio.csv`

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

DATE = "20260406"
REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "script/res" / f"{DATE}_nxdomain_ratio_stats.csv").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside dnsnsec26 or set REPO_ROOT explicitly.")
RES_DIR = REPO_ROOT / "script/res"
OUTPUT_DIR = REPO_ROOT / "notebook/results"
WWW_SUMMARY = RES_DIR / f"{DATE}_www_nxdomain_ratio.csv"
ZONES = ["com", "net", "org", "umbrella", "tranco", "radar", "majestic"]
LABELS = {"com": ".com", "net": ".net", "org": ".org",
          "umbrella": "Umbrella", "tranco": "Tranco",
          "radar": "Radar", "majestic": "Majestic"}

## Load and compute status proportions
Normalize both summary schemas into NODATA count, NXDOMAIN count, and total queries. Checks enforce nonnegative integer counts, positive denominators, complete dataset/mechanism coverage, and `NODATA + NXDOMAIN = total`.

The WWW file repeats its total on each status row. Check that these totals agree and use one total per dataset/mechanism, rather than adding the repeated totals. Its saved `perc` values are checked against `count / total`.

The WWW `.com` NSEC summary has 1,463,428 total queries, one more than Table 1's 1,463,427 domain count. This notebook uses the response-status summary's denominator, consistently with the Table 4 calculation; the one-query difference does not change the displayed percentage.

In [5]:
def load_status_summary(path, population):
    frame = pd.read_csv(path).rename(columns={
        "0": "nodata_count", "3": "nxdomain_count", "total": "total_queries",
        "Zone": "zone", "Type": "mechanism",
    })
    if population == "WWW":
        import math
        required_long = {"status", "count", "total_queries", "perc", "zone", "type"}
        if not required_long.issubset(frame.columns):
            raise ValueError(f"{path.name}: missing WWW summary columns.")
        frame["mechanism"] = frame["type"].str.upper()
        group_keys = ["zone", "mechanism"]
        expected_status_rows = {(zone, mechanism, status)
                                for zone in ZONES for mechanism in ("NSEC", "NSEC3")
                                for status in ("NODATA", "NXDOMAIN")}
        if (frame.duplicated(group_keys + ["status"]).any()
                or set(map(tuple, frame[group_keys + ["status"]].to_numpy())) != expected_status_rows):
            raise ValueError("Expected one NODATA and one NXDOMAIN row per WWW dataset/mechanism.")
        for column in ["count", "total_queries"]:
            values = pd.to_numeric(frame[column], errors="raise")
            if values.isna().any() or values.lt(0).any() or (values % 1 != 0).any():
                raise ValueError(f"{path.name}: {column} must contain nonnegative integers.")
            frame[column] = values.astype("int64")
        if frame["total_queries"].le(0).any():
            raise ValueError("WWW denominators must be positive.")
        if not frame.groupby(group_keys)["total_queries"].nunique().eq(1).all():
            raise ValueError("WWW status rows have inconsistent totals.")
        saved = pd.to_numeric(frame["perc"], errors="raise")
        computed = frame["count"] / frame["total_queries"]
        if not all(math.isclose(a, b, rel_tol=1e-12, abs_tol=1e-15)
                   for a, b in zip(saved, computed)):
            raise ValueError("WWW saved fractions differ from count / total.")
        counts = frame.pivot(index=group_keys, columns="status", values="count")
        counts = counts.rename(columns={"NODATA": "nodata_count", "NXDOMAIN": "nxdomain_count"})
        totals = frame.groupby(group_keys)["total_queries"].first()
        frame = counts.join(totals).reset_index()

    required = ["zone", "mechanism", "nodata_count", "nxdomain_count", "total_queries"]
    if not set(required).issubset(frame.columns):
        raise ValueError(f"{path.name}: missing required columns.")
    frame = frame[required].copy()
    expected = {(zone, mechanism) for zone in ZONES for mechanism in ("NSEC", "NSEC3")}
    if frame.duplicated(["zone", "mechanism"]).any() or set(map(tuple, frame[["zone", "mechanism"]].to_numpy())) != expected:
        raise ValueError(f"{path.name}: expected one row per dataset/mechanism.")
    for column in ["nodata_count", "nxdomain_count", "total_queries"]:
        values = pd.to_numeric(frame[column], errors="raise")
        if values.isna().any() or values.lt(0).any() or (values % 1 != 0).any():
            raise ValueError(f"{path.name}: {column} must contain nonnegative integers.")
        frame[column] = values.astype("int64")
    if frame["total_queries"].le(0).any():
        raise ValueError("Query denominators must be positive.")
    if not (frame["nodata_count"] + frame["nxdomain_count"]).eq(frame["total_queries"]).all():
        raise ValueError("NODATA and NXDOMAIN counts must sum to total_queries.")
    frame["population"] = population
    frame["nxdomain_pct"] = 100 * frame["nxdomain_count"] / frame["total_queries"]
    frame["nodata_pct"] = 100 * frame["nodata_count"] / frame["total_queries"]
    return frame


source_stats = load_status_summary(RES_DIR / f"{DATE}_nxdomain_ratio_stats.csv", "Source")
source_totals = pd.read_csv(RES_DIR / f"{DATE}_domcnt_stats.csv").set_index("zone")
for mechanism in ("NSEC", "NSEC3"):
    observed = source_stats.loc[source_stats["mechanism"].eq(mechanism)].set_index("zone")["total_queries"].loc[ZONES]
    expected = source_totals.loc[ZONES, mechanism.lower()]
    pd.testing.assert_series_equal(observed, expected, check_names=False)

www_stats = load_status_summary(WWW_SUMMARY, "WWW")
stats = pd.concat([source_stats, www_stats], ignore_index=True)
print(f"Loaded and validated source and WWW summaries for {DATE}.")

Loaded and validated source and WWW summaries for 20260406.


## Recreated Table 4
Column order follows the paper: Source NSEC/NSEC3, followed by WWW NSEC/NSEC3. Every cell is calculated from its corresponding summary counts.

In [6]:
paper_columns = pd.MultiIndex.from_tuples([
    ("Source", "NSEC"), ("Source", "NSEC3"),
    ("WWW", "NSEC"), ("WWW", "NSEC3"),
], names=["Population", "Mechanism"])
table4 = stats.pivot(index="zone", columns=["population", "mechanism"], values="nxdomain_pct")
table4 = table4.reindex(index=ZONES, columns=paper_columns)
table4.index = table4.index.map(LABELS)
table4.index.name = "Dataset"
if table4.isna().any().any():
    raise ValueError("Expected all 28 Table 4 percentages to be available.")
formatted_table4 = table4.apply(lambda column: column.map(lambda value: f"{value:.2f}%"))
display(formatted_table4)

Population  Source             WWW        
Mechanism     NSEC   NSEC3    NSEC   NSEC3
Dataset                                   
.com        41.50%  89.11%  52.43%  94.56%
.net        31.77%  88.32%  37.78%  93.43%
.org        35.03%  89.42%  41.63%  94.39%
Umbrella    19.41%  82.34%  18.48%  92.13%
Tranco      21.25%  77.14%  21.52%  90.57%
Radar       16.46%  73.48%  18.11%  88.37%
Majestic    21.39%  77.14%  25.34%  92.10%

## Export
- `table4_stats.csv`: source and WWW counts, denominators, NXDOMAIN percentages, and NODATA percentages.
- `table4_formatted.csv`: reproduced Table 4 with two header rows.
- `table4_paper_differences.csv`: calculated minus published percentages in percentage points, after rounding to two decimals.

Rerunning this cell replaces these exports.

In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
stats.to_csv(OUTPUT_DIR / "table4_stats.csv", index=False)
print(f"Saved Table 4 results to {OUTPUT_DIR}")

Saved Table 4 results to /u/yz6me/dnsnsec26/notebook/results
